## Dataset Architecture & Feature Schema

This dataset evaluates coastal, inland lake, and riverfront resorts across Kenya by leveraging spatial data extracted via the **Google Places API**. It measures the impact of surrounding environmental noise vectors on guest satisfaction (`GoogleRating`).

The dataset is organized across **13 core columns** grouped into 3 key analytical categories:

### 1. Core Identification & Spatial Context
* **`ResortName`**: Official name of the resort property.
* **`Address`**: Physical address or geographic location string.
* **`WaterBodyType`**: Classification of adjacent water body (*Marine Coast*, *Inland Lake*, or *Safari River*).
* **`Location`**: Regional geographic zone (e.g., *Coast - Diani & South*, *Inland Lake - Naivasha*, *Safari River Lodge*).
* **`Latitude`**: Exact geographic latitude coordinate.
* **`Longitude`**: Exact geographic longitude coordinate.

### 2. Environmental Noise Footprint (1km Radius)
* **`BarsCount`**: Total count of active bars/pubs within a 1km radius.
* **`NightClubsCount`**: Total count of active nightlife venues within a 1km radius.
* **`TransitCount`**: Total count of bus stops, matatu stages, and transport hubs within a 1km radius.
* **`WorshipCount`**: Total count of active places of worship (churches, mosques) within a 1km radius.
* **`TotalNoiseSources`**: Composite sum of all surrounding noise vectors, functioning as an objective **Noise Nuisance Index**.

### 3. Visitor Satisfaction Metrics
* **`GoogleRating`**: Aggregated average guest review rating out of 5.0 stars from Google Places.

### Environment Setup & Library Imports

In [1]:
import os
import pandas as pd
import googlemaps
from dotenv import load_dotenv

# Loading secure environment variables from .env file
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if not API_KEY:
    raise ValueError("❌ Error: GOOGLE_MAPS_API_KEY not found in your .env file!")

# Initialize the Google Maps client
gmaps = googlemaps.Client(key=API_KEY)
print("✅ Environment initialized and Google Maps client ready.")

✅ Environment initialized and Google Maps client ready.


### Regional Discovery Pipeline (Comprehensive Scraper)

In [2]:
# Search queries covering coastal marine waters and inland water bodies (Lakes & Rivers)
expanded_water_search_queries = [
    # Coastal Marine Waters
    "resorts in Diani Beach Kenya",
    "beach resorts in Mombasa Shanzu Nyali",
    "resorts in Watamu and Malindi Kenya",
    "hotels in Lamu Archipelago Shela",
    # Inland Lakes (Rift Valley & Western Kenya)
    "resorts around Lake Naivasha Kenya",
    "hotels near Lake Nakuru Kenya",
    "lodges around Lake Elementaita",
    "waterfront hotels in Kisumu Lake Victoria",
    # Major Safari River Lodges
    "lodges along Mara River Masai Mara",
    "river lodges Samburu Ewaso Nyiro"
]

all_water_resorts_dict = {}

print("🌍 Scanning and pulling water-proximate resorts across Kenya (Coastal & Inland)...")

for query in expanded_water_search_queries:
    print(f"🔍 Executing search query: '{query}'...")
    try:
        response = gmaps.places(query=query)
        results = response.get("results", [])
        
        for place in results:
            name = place.get("name")
            place_id = place.get("place_id")
            
            if place_id and place_id not in all_water_resorts_dict:
                address = place.get("formatted_address", "")
                rating = place.get("rating", None)
                lat = place.get("geometry", {}).get("location", {}).get("lat", None)
                lng = place.get("geometry", {}).get("location", {}).get("lng", None)
                
                # Dynamic Region / Water Body Categorization
                region = "Other Water Body"
                q_lower = query.lower()
                if "diani" in q_lower or "galu" in q_lower:
                    region = "Coast - Diani & South"
                elif "mombasa" in q_lower or "shanzu" in q_lower or "nyali" in q_lower:
                    region = "Coast - Mombasa & North"
                elif "watamu" in q_lower or "malindi" in q_lower:
                    region = "Coast - Watamu & Malindi"
                elif "lamu" in q_lower or "shela" in q_lower:
                    region = "Coast - Lamu"
                elif "naivasha" in q_lower:
                    region = "Inland Lake - Naivasha"
                elif "nakuru" in q_lower:
                    region = "Inland Lake - Nakuru"
                elif "elementaita" in q_lower:
                    region = "Inland Lake - Elementaita"
                elif "kisumu" in q_lower or "victoria" in q_lower:
                    region = "Inland Lake - Victoria"
                elif "mara" in q_lower or "samburu" in q_lower or "river" in q_lower:
                    region = "Safari River Lodge"

                all_water_resorts_dict[place_id] = {
                    "ResortName": name,
                    "Address": address,
                    "WaterBodyType": "Marine Coast" if "Coast" in region else ("Inland Lake" if "Lake" in region else "Safari River"),
                    "Location": region,
                    "Latitude": lat,
                    "Longitude": lng,
                    "GoogleRating": rating,
                    "BeachCrowdIndex": None,
                    "QuietnessScore": None,
                    "PrivateBeachAccess": None,
                    "SatisfactionPercent": None
                }
    except Exception as e:
        print(f"⚠️ Error with query '{query}': {e}")

# Converting to DataFrame
df_resorts = pd.DataFrame(list(all_water_resorts_dict.values()))
print(f"\n🚀 Successfully compiled {len(df_resorts)} water-proximate properties across Kenya.")
df_resorts.head()

🌍 Scanning and pulling water-proximate resorts across Kenya (Coastal & Inland)...
🔍 Executing search query: 'resorts in Diani Beach Kenya'...
🔍 Executing search query: 'beach resorts in Mombasa Shanzu Nyali'...
🔍 Executing search query: 'resorts in Watamu and Malindi Kenya'...
🔍 Executing search query: 'hotels in Lamu Archipelago Shela'...
🔍 Executing search query: 'resorts around Lake Naivasha Kenya'...
🔍 Executing search query: 'hotels near Lake Nakuru Kenya'...
🔍 Executing search query: 'lodges around Lake Elementaita'...
🔍 Executing search query: 'waterfront hotels in Kisumu Lake Victoria'...
🔍 Executing search query: 'lodges along Mara River Masai Mara'...
🔍 Executing search query: 'river lodges Samburu Ewaso Nyiro'...

🚀 Successfully compiled 170 water-proximate properties across Kenya.


,ResortName,Address,WaterBodyType,Location,Latitude,Longitude,GoogleRating,BeachCrowdIndex,QuietnessScore,PrivateBeachAccess,SatisfactionPercent
0,Diani Reef Beach Resort & Spa,"Diani Beach Road, Diani Beach, Kenya",Marine Coast,Coast - Diani & South,-4.278903,39.593257,4.3,None,None,None,None
1,Swahili Beach Resort,"Ukunda, Diani Beach Road, Kwale, Kenya",Marine Coast,Coast - Diani & South,-4.292183,39.586675,4.4,None,None,None,None
2,Southern Palms Beach Resort,"Plot LR 13430, Diani Beach Road Diani, Ukunda,...",Marine Coast,Coast - Diani & South,-4.268264,39.597005,4.3,None,None,None,None
3,Diamonds Leisure Beach & Golf Resort,"Diani Road, South, Coast, Kwale, Kenya",Marine Coast,Coast - Diani & South,-4.281405,39.592070,4.4,None,None,None,None
4,Leopard Beach Resort and Spa,"Diani Beach, Kenya",Marine Coast,Coast - Diani & South,-4.284109,39.589231,4.4,None,None,None,None


### Categorical Noise Breakdown

In [3]:
def assess_detailed_noise_sources(lat, lng):
    """
    Scans a 1km radius around resort coordinates and returns individual 
    counts for specific noise/disruptor categories.
    """
    if pd.isna(lat) or pd.isna(lng):
        return {"BarsCount": 0, "NightClubsCount": 0, "TransitCount": 0, "WorshipCount": 0}
        
    noise_categories = {
        "BarsCount": "bar",
        "NightClubsCount": "night_club",
        "TransitCount": "transit_station",
        "WorshipCount": "church"
    }
    
    counts = {}
    for col_name, place_type in noise_categories.items():
        try:
            results = gmaps.places_nearby(
                location=(lat, lng),
                radius=1000, # 1km radius
                type=place_type
            )
            counts[col_name] = len(results.get("results", []))
        except Exception:
            counts[col_name] = 0
            
    # Checking mosques separately and add to worship count
    try:
        mosques = gmaps.places_nearby(location=(lat, lng), radius=1000, keyword="mosque")
        counts["WorshipCount"] += len(mosques.get("results", []))
    except Exception:
        pass
        
    return counts

print("🎧 Scanning detailed noise source categories for expanded properties...")

# Applying the function and expanding the dictionary into separate columns
noise_df_expanded = df_resorts.apply(
    lambda row: pd.Series(assess_detailed_noise_sources(row['Latitude'], row['Longitude'])), 
    axis=1
)

# Joining the new category columns back into your main DataFrame
df_resorts = pd.concat([df_resorts, noise_df_expanded], axis=1)

# Creating total combined noise score column
df_resorts['TotalNoiseSources'] = (
    df_resorts['BarsCount'] + 
    df_resorts['NightClubsCount'] + 
    df_resorts['TransitCount'] + 
    df_resorts['WorshipCount']
)

# Dropping any columns that are completely empty
df_resorts = df_resorts.dropna(axis=1, how='all')

# Exporting updated dataframe
output_dir = "Data"
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, "kenya_coastal_resorts_spatial_analysis.csv")
json_path = os.path.join(output_dir, "kenya_coastal_resorts_spatial_analysis.json")

df_resorts.to_csv(csv_path, index=False)
df_resorts.to_json(json_path, orient="records", indent=4)

print(f"✅ Detailed noise metrics computed and master dataset successfully saved to: {csv_path}")
df_resorts[['ResortName', 'WaterBodyType', 'BarsCount', 'NightClubsCount', 'TransitCount', 'WorshipCount', 'TotalNoiseSources']].head()

🎧 Scanning detailed noise source categories for expanded properties...
✅ Detailed noise metrics computed and master dataset successfully saved to: Data\kenya_coastal_resorts_spatial_analysis.csv


,ResortName,WaterBodyType,BarsCount,NightClubsCount,TransitCount,WorshipCount,TotalNoiseSources
0,Diani Reef Beach Resort & Spa,Marine Coast,11,3,0,21,35
1,Swahili Beach Resort,Marine Coast,11,1,0,23,35
2,Southern Palms Beach Resort,Marine Coast,10,2,0,21,33
3,Diamonds Leisure Beach & Golf Resort,Marine Coast,9,3,0,22,34
4,Leopard Beach Resort and Spa,Marine Coast,9,1,0,22,32


### Exporting My Master Dataset

In [4]:
output_dir = "Data"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "kenya_water_resorts_spatial_analysis.csv")
json_path = os.path.join(output_dir, "kenya_water_resorts_spatial_analysis.json")

df_resorts.to_csv(csv_path, index=False)
df_resorts.to_json(json_path, orient="records", indent=4)

print(f"📁 Master dataset successfully exported to: {csv_path}")

📁 Master dataset successfully exported to: Data\kenya_water_resorts_spatial_analysis.csv


### Quick DataFrame Inspection

In [5]:
# 1. Statistical summary of the composite noise index
print("=" * 60)
print("📊 SUMMARY STATS: TOTAL NOISE SOURCES")
print("=" * 60)
print(df_resorts['TotalNoiseSources'].describe())

📊 SUMMARY STATS: TOTAL NOISE SOURCES
count    170.000000
mean      32.529412
std       16.777407
min       20.000000
25%       20.000000
50%       26.000000
75%       35.000000
max       97.000000
Name: TotalNoiseSources, dtype: float64


In [6]:
# 2. Top 5 resorts with the HIGHEST noise footprint (Least Secluded)
print("\n" + "=" * 60)
print("🔊 TOP 5 LOUDEST PROPERTIES")
print("=" * 60)
loudest_properties = df_resorts.sort_values(
    by='TotalNoiseSources', ascending=False
)[['ResortName', 'WaterBodyType', 'Location', 'TotalNoiseSources']].head(5)

print(loudest_properties.to_string(index=False))


🔊 TOP 5 LOUDEST PROPERTIES
                   ResortName WaterBodyType               Location  TotalNoiseSources
Sarova Imperial Hotel, Kisumu   Inland Lake Inland Lake - Victoria                 97
             Mara River Lodge  Safari River     Safari River Lodge                 88
               Selella Resort   Inland Lake Inland Lake - Victoria                 77
 Acacia Premier Hotel, Kisumu   Inland Lake Inland Lake - Victoria                 77
    The Ole-Ken Hotel, Nakuru   Inland Lake   Inland Lake - Nakuru                 75


In [7]:
# 3. Top 5 resorts with the LOWEST noise footprint (Most Secluded)
print("\n" + "=" * 60)
print("🌿 TOP 5 QUIETEST PROPERTIES")
print("=" * 60)
quietest_properties = df_resorts.sort_values(
    by='TotalNoiseSources', ascending=True
)[['ResortName', 'WaterBodyType', 'Location', 'TotalNoiseSources']].head(5)

print(quietest_properties.to_string(index=False))


🌿 TOP 5 QUIETEST PROPERTIES
                                    ResortName WaterBodyType                 Location  TotalNoiseSources
Kivulini Beach Resort (Kivulini Luxury Resort)  Marine Coast Coast - Watamu & Malindi                 20
                               Manda Bay Lodge  Marine Coast             Coast - Lamu                 20
                         The Majlis Hotel Lamu  Marine Coast             Coast - Lamu                 20
                        Flamingo Villas Resort  Marine Coast Coast - Watamu & Malindi                 20
                                Mangrove House  Marine Coast             Coast - Lamu                 20


### Data Inspection

In [8]:
# 1. Load your cleaned expanded dataset
df = pd.read_csv("Data/kenya_coastal_resorts_spatial_analysis.csv")

# 2. Define active noise source metrics
noise_cols = ['BarsCount', 'NightClubsCount', 'TransitCount', 'WorshipCount', 'TotalNoiseSources']

# ----------------------------------------------------------------------
# A. AVERAGE NOISE SOURCES BY WATER BODY TYPE (Marine vs Lake vs River)
# ----------------------------------------------------------------------
print("=" * 70)
print("🌊 AVERAGE NOISE SOURCE COUNTS BY WATER BODY TYPE")
print("=" * 70)
avg_noise_by_water_type = df.groupby('WaterBodyType')[noise_cols].mean().reset_index()
avg_noise_by_water_type[noise_cols] = avg_noise_by_water_type[noise_cols].round(2)
avg_noise_by_water_type = avg_noise_by_water_type.rename(
    columns={col: f"Avg_{col}" for col in noise_cols if col != 'WaterBodyType'}
)
print(avg_noise_by_water_type.to_string(index=False))

# ----------------------------------------------------------------------
# B. AVERAGE NOISE SOURCES BY REGIONAL LOCATION
# ----------------------------------------------------------------------
print("\n" + "=" * 70)
print("📊 AVERAGE NOISE SOURCE COUNTS BY LOCATION")
print("=" * 70)
avg_noise_by_location = df.groupby(['WaterBodyType', 'Location'])[noise_cols].mean().reset_index()
avg_noise_by_location[noise_cols] = avg_noise_by_location[noise_cols].round(2)
avg_noise_by_location = avg_noise_by_location.rename(
    columns={col: f"Avg_{col}" for col in noise_cols if col not in ['WaterBodyType', 'Location']}
)
print(avg_noise_by_location.to_string(index=False))

🌊 AVERAGE NOISE SOURCE COUNTS BY WATER BODY TYPE
WaterBodyType  Avg_BarsCount  Avg_NightClubsCount  Avg_TransitCount  Avg_WorshipCount  Avg_TotalNoiseSources
  Inland Lake           7.52                 2.25              1.04             27.76                  38.58
 Marine Coast           5.63                 0.82              0.15             22.00                  28.60
 Safari River           0.87                 0.35              0.87             20.87                  22.96

📊 AVERAGE NOISE SOURCE COUNTS BY LOCATION
WaterBodyType                  Location  Avg_BarsCount  Avg_NightClubsCount  Avg_TransitCount  Avg_WorshipCount  Avg_TotalNoiseSources
  Inland Lake Inland Lake - Elementaita           3.70                 0.35              0.20             23.05                  27.30
  Inland Lake    Inland Lake - Naivasha           2.95                 0.75              0.45             23.25                  27.40
  Inland Lake      Inland Lake - Nakuru          13.10             

In [9]:
# ----------------------------------------------------------------------
# C. LOCATION RANKINGS (LOUDEST TO QUIETEST)
# ----------------------------------------------------------------------
location_ranking = df.groupby('Location')['TotalNoiseSources'].mean().reset_index()
location_ranking = location_ranking.sort_values(by='TotalNoiseSources', ascending=False).reset_index(drop=True)
location_ranking['TotalNoiseSources'] = location_ranking['TotalNoiseSources'].round(2)
location_ranking = location_ranking.rename(columns={'TotalNoiseSources': 'Avg_TotalNoiseSources'})

print("\n" + "=" * 70)
print("🔊 LOCATIONS RANKED: LOUDEST TO QUIETEST (Average Total Noise)")
print("=" * 70)
print(location_ranking.to_string(index=False))

# Highlight extremes
loudest_loc = location_ranking.iloc[0]['Location']
quietest_loc = location_ranking.iloc[-1]['Location']

print("\n" + "=" * 70)
print(f"🏆 Loudest Region: {loudest_loc}")
print(f"🌿 Quietest Region: {quietest_loc}")
print("=" * 70)


🔊 LOCATIONS RANKED: LOUDEST TO QUIETEST (Average Total Noise)
                 Location  Avg_TotalNoiseSources
     Inland Lake - Nakuru                  52.15
   Inland Lake - Victoria                  47.45
  Coast - Mombasa & North                  34.14
    Coast - Diani & South                  33.75
   Inland Lake - Naivasha                  27.40
Inland Lake - Elementaita                  27.30
 Coast - Watamu & Malindi                  26.90
             Coast - Lamu                  23.20
       Safari River Lodge                  22.96

🏆 Loudest Region: Inland Lake - Nakuru
🌿 Quietest Region: Safari River Lodge


In [10]:
# 3. Find the hotel(s) with the MINIMUM total noise sources
min_noise_val = df['TotalNoiseSources'].min()
min_noise_hotels = df[df['TotalNoiseSources'] == min_noise_val]

print("\n" + "=" * 60)
print(f"🌿 QUIETEST PROPERTIES (Min Total Noise Sources: {min_noise_val})")
print("=" * 60)
print(min_noise_hotels[['ResortName', 'WaterBodyType', 'Location', 'TotalNoiseSources', 'BarsCount', 'NightClubsCount', 'TransitCount', 'WorshipCount']].to_string(index=False))


🌿 QUIETEST PROPERTIES (Min Total Noise Sources: 20)
                                                                  ResortName WaterBodyType                  Location  TotalNoiseSources  BarsCount  NightClubsCount  TransitCount  WorshipCount
                              Kivulini Beach Resort (Kivulini Luxury Resort)  Marine Coast  Coast - Watamu & Malindi                 20          0                0             0            20
                                                      Flamingo Villas Resort  Marine Coast  Coast - Watamu & Malindi                 20          0                0             0            20
                                                       The Majlis Hotel Lamu  Marine Coast              Coast - Lamu                 20          0                0             0            20
                                                              Mangrove House  Marine Coast              Coast - Lamu                 20          0                0             0  

In [11]:
# 4. Find the hotel(s) with the MAXIMUM total noise sources
max_noise_val = df['TotalNoiseSources'].max()
max_noise_hotels = df[df['TotalNoiseSources'] == max_noise_val]

print("\n" + "=" * 60)
print(f"🔊 LOUDEST PROPERTIES (Max Total Noise Sources: {max_noise_val})")
print("=" * 60)
print(max_noise_hotels[['ResortName', 'WaterBodyType', 'Location', 'TotalNoiseSources', 'BarsCount', 'NightClubsCount', 'TransitCount', 'WorshipCount']].to_string(index=False))


🔊 LOUDEST PROPERTIES (Max Total Noise Sources: 97)
                   ResortName WaterBodyType               Location  TotalNoiseSources  BarsCount  NightClubsCount  TransitCount  WorshipCount
Sarova Imperial Hotel, Kisumu   Inland Lake Inland Lake - Victoria                 97         20               20            18            39


In [12]:
# 3. Aggregate noise levels and ratings grouped by region
print("\n" + "=" * 50)
print("📍 REGIONAL BREAKDOWN BY WATER BODY & LOCATION")
print("=" * 50)
location_summary = df.groupby(['WaterBodyType', 'Location'])[['TotalNoiseSources', 'GoogleRating']].agg(
    ['mean', 'min', 'max', 'count']
)
print(location_summary)


📍 REGIONAL BREAKDOWN BY WATER BODY & LOCATION
                                        TotalNoiseSources                \
                                                     mean min max count   
WaterBodyType Location                                                    
Inland Lake   Inland Lake - Elementaita         27.300000  20  67    20   
              Inland Lake - Naivasha            27.400000  20  73    20   
              Inland Lake - Nakuru              52.150000  20  75    20   
              Inland Lake - Victoria            47.450000  24  97    20   
Marine Coast  Coast - Diani & South             33.750000  24  40    20   
              Coast - Lamu                      23.200000  20  29    20   
              Coast - Mombasa & North           34.142857  25  47     7   
              Coast - Watamu & Malindi          26.900000  20  54    20   
Safari River  Safari River Lodge                22.956522  20  88    23   

                                        GoogleRating

In [13]:
# 4. Calculate statistical correlation between environmental noise and guest ratings
correlation = df['TotalNoiseSources'].corr(df['GoogleRating'])
print("\n" + "=" * 50)
print("📈 CORRELATION ANALYSIS (KENYA WATER-PROXIMATE RESORTS)")
print("=" * 50)
print(f"Correlation (Total Noise Sources vs. Google Rating): {correlation:.2f}")


📈 CORRELATION ANALYSIS (KENYA WATER-PROXIMATE RESORTS)
Correlation (Total Noise Sources vs. Google Rating): -0.37
